# 03 — EWOA Feature Selection + KNN

**Cel**: Reprodukcja artykułu — EWOA selekcja cech → KNN klasyfikacja.

Pipeline:
1. Załaduj dane CIC-MalMem-2022 (55 cech)
2. Uruchom EWOA (20 wielorybów, 30 iteracji)
3. Uruchom WOA (baseline do porównania)
4. Porównaj wyniki z artykułem (accuracy 99.987%, avg 3.97 cech)

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..'))

import numpy as np

from src.data.loader import load_raw_data, preprocess, make_splits
from src.algorithms import EWOA, WOA
from src.evaluation.metrics import evaluate_knn, compare_algorithms
from src.evaluation.visualization import (
    plot_convergence, plot_feature_selection,
    plot_accuracy_comparison, plot_confusion_matrix,
    plot_n_features_comparison,
)
from src.utils.config import SEED

np.random.seed(SEED)

## 1. Ładowanie i podział danych

In [ ]:
df = load_raw_data()
X, y = preprocess(df, mode='multiclass')
X_train, X_test, y_train, y_test = make_splits(X, y)

feature_names = [c for c in df.columns if c not in ('Category', 'Class')]
print(f'Cechy ({len(feature_names)}): {feature_names[:5]} ...')

## 2. EWOA — Enhanced Whale Optimization Algorithm

In [ ]:
ewoa = EWOA(n_whales=20, max_iter=30, n_neighbors=5, alpha=0.01, seed=SEED)

t0 = time.time()
ewoa_result = ewoa.optimize(X_train, y_train)
ewoa_time = time.time() - t0

print(f'\n--- EWOA ---')
print(f'Wybrane cechy ({ewoa_result["n_selected"]}): {ewoa_result["selected_features"]}')
print(f'Fitness: {ewoa_result["best_fitness"]:.6f}')
print(f'Czas: {ewoa_time:.1f} s')

if feature_names:
    selected_names = [feature_names[i] for i in ewoa_result['selected_features']]
    print(f'Nazwy cech: {selected_names}')

## 3. WOA — bazowy Whale Optimization (porównanie)

In [ ]:
woa = WOA(n_whales=20, max_iter=30, n_neighbors=5, alpha=0.01, seed=SEED)

t0 = time.time()
woa_result = woa.optimize(X_train, y_train)
woa_time = time.time() - t0

print(f'\n--- WOA ---')
print(f'Wybrane cechy ({woa_result["n_selected"]}): {woa_result["selected_features"]}')
print(f'Fitness: {woa_result["best_fitness"]:.6f}')
print(f'Czas: {woa_time:.1f} s')

## 4. Ewaluacja na zbiorze testowym

In [ ]:
print('=== EWOA + KNN ===')
ewoa_eval = evaluate_knn(
    X_train, y_train, X_test, y_test,
    selected_features=ewoa_result['selected_features'],
    n_neighbors=5, algorithm_name='EWOA'
)

print('\n=== WOA + KNN ===')
woa_eval = evaluate_knn(
    X_train, y_train, X_test, y_test,
    selected_features=woa_result['selected_features'],
    n_neighbors=5, algorithm_name='WOA'
)

print('\n=== KNN baseline (55 cech) ===')
baseline_eval = evaluate_knn(
    X_train, y_train, X_test, y_test,
    selected_features=list(range(55)),
    n_neighbors=5, algorithm_name='KNN-all'
)

## 5. Tabela porównawcza

In [ ]:
all_results = [ewoa_eval, woa_eval, baseline_eval]
comparison_df = compare_algorithms(all_results)
comparison_df

## 6. Wizualizacje

In [ ]:
# Krzywe zbieżności
plot_convergence({
    'EWOA': ewoa_result['convergence'],
    'WOA': woa_result['convergence'],
})

In [ ]:
# Heatmapa selekcji cech
plot_feature_selection(
    {'EWOA': ewoa_result['selected_features'], 'WOA': woa_result['selected_features']},
    feature_names=feature_names,
)

In [ ]:
# Porównanie accuracy
plot_accuracy_comparison(all_results)

In [ ]:
# Porównanie liczby cech
plot_n_features_comparison(all_results)

In [ ]:
# Confusion matrix — EWOA
from sklearn.neighbors import KNeighborsClassifier

knn_final = KNeighborsClassifier(n_neighbors=5)
knn_final.fit(X_train[:, ewoa_result['selected_features']], y_train)
y_pred_ewoa = knn_final.predict(X_test[:, ewoa_result['selected_features']])

plot_confusion_matrix(y_test, y_pred_ewoa, algorithm_name='EWOA', save_name='confusion_ewoa')

## 7. Porównanie z artykułem

| Metryka | Artykuł (EWOA+KNN) | Nasz wynik |
|---------|:---:|:---:|
| Accuracy | 99.987% | **patrz wyżej** |
| Avg. features | 3.97 | **patrz wyżej** |
| Avg. time | 43.19 s | **patrz wyżej** |

In [ ]:
print('Porównanie z artykułem:')
print(f'  Accuracy:  nasz={ewoa_eval["accuracy"]:.5f}  vs  artykuł=0.99987')
print(f'  N cech:    nasz={ewoa_eval["n_features"]}      vs  artykuł=3.97 (avg)')
print(f'  Czas:      nasz={ewoa_time:.1f}s    vs  artykuł=43.19s')